# Lab 03 — SDKs, streaming, tool use
Stream a response, measure TTFT, then stream a tool call.
Cost math: 30 input + 400 output on claude-sonnet-4-6 => (30/1e6)*3 + (400/1e6)*15 = $0.00609 ~ $0.01 per full run.

In [ ]:
!uv add anthropic openai

In [ ]:
import os, time
from anthropic import Anthropic
client = Anthropic()
t0 = time.perf_counter()
with client.messages.stream(model='claude-sonnet-4-6', max_tokens=400, temperature=0,
        messages=[{'role':'user','content':'List 5 arXiv papers on ReAct agents.'}]) as s:
    first = None
    for text in s.text_stream:
        if first is None and text.strip():
            first = time.perf_counter() - t0
        print(text, end='', flush=True)
    final = s.get_final_message()
print(f'\n--- ttft={first*1000:.0f}ms tokens_out={final.usage.output_tokens}')

In [ ]:
# Streaming a tool call: accumulate deltas, parse once at stop.
TOOL = {'name':'get_weather','description':'weather for a city',
        'input_schema':{'type':'object','properties':{'city':{'type':'string'}},'required':['city']}}
with client.messages.stream(model='claude-sonnet-4-6', max_tokens=200, temperature=0,
        tools=[TOOL], messages=[{'role':'user','content':"What's the weather in Paris?"}]) as s:
    for _ in s: pass
    msg = s.get_final_message()
tool = next(b for b in msg.content if b.type=='tool_use')
print('tool input:', tool.input)